# Acceptance: contact-survey matrices

**This is a sign-off page, not a tutorial.** It exercises roadmap step 17
(work package `WP9`): `summer4.epi.ContactMatrix`, which loads, validates and
inspects an empirical contact-survey matrix, and `SettingStack`, the
per-setting stack these datasets ship as. Nothing here asserts; the automated
checks live in `tests/test_contact_matrices.py`.

Run every cell top to bottom (a few seconds). Sign off only if every
**What to check** box holds.

| | You are checking that… |
|---|---|
| A | a published survey table loads in participant-by-contact orientation, and the row and column sums mean different things |
| B | a long frame and a per-setting frame load to the same matrix |
| C | bad input is refused with the offending value named |
| D | reciprocity error and mean contacts are measurable against a population |

**Orientation, fixed once:** `values[i, j]` is the average number of contacts a
member of band `i` (the *participant*) has with members of band `j` (the
*contact*), per person per day.

**Where the data come from.** summer4 ships no survey data (roadmap step 17,
decided 2026-09-29): it ships loaders, and each page embeds the small public
extract it needs, so nothing is fetched when the docs build. The matrix below is
the POLYMOD Great Britain all-settings matrix from Mossong *et al.* (2008),
"Social contacts and mixing patterns relevant to the spread of infectious
diseases", *PLoS Medicine* 5(3): e74, supporting table
[doi:10.1371/journal.pmed.0050074.st005](https://doi.org/10.1371/journal.pmed.0050074.st005),
published under the Creative Commons Attribution licence. It is carried here as
transcribed in the [summer textbook](https://github.com/monash-emu/summer-textbook)
(BSD-2-Clause, Copyright (c) 2022, monash-emu) chapter 17, together with that
textbook's United Kingdom 2006 population by age.

In [ ]:
import math

import numpy as np
import pandas as pd
import plotly.io as pio

from summer4.epi import ContactMatrix, SettingStack

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

# POLYMOD Great Britain, as published: rows are the AGE OF CONTACT, columns the
# AGE GROUP OF PARTICIPANT (the transpose of summer4's orientation).
PUBLISHED = np.array(
    [
        [1.92, 0.65, 0.41, 0.24, 0.46, 0.73, 0.67, 0.83, 0.24, 0.22, 0.36, 0.20, 0.20, 0.26, 0.13],
        [0.95, 6.64, 1.09, 0.73, 0.61, 0.75, 0.95, 1.39, 0.90, 0.16, 0.30, 0.22, 0.50, 0.48, 0.20],
        [0.48, 1.31, 6.85, 1.52, 0.27, 0.31, 0.48, 0.76, 1.00, 0.69, 0.32, 0.44, 0.27, 0.41, 0.33],
        [0.33, 0.34, 1.03, 6.71, 1.58, 0.73, 0.42, 0.56, 0.85, 1.16, 0.70, 0.30, 0.20, 0.48, 0.63],
        [0.45, 0.30, 0.22, 0.93, 2.59, 1.49, 0.75, 0.63, 0.77, 0.87, 0.88, 0.61, 0.53, 0.37, 0.33],
        [0.79, 0.66, 0.44, 0.74, 1.29, 1.83, 0.97, 0.71, 0.74, 0.85, 0.88, 0.87, 0.67, 0.74, 0.33],
        [0.97, 1.07, 0.62, 0.50, 0.88, 1.19, 1.67, 0.89, 1.02, 0.91, 0.92, 0.61, 0.76, 0.63, 0.27],
        [1.02, 0.98, 1.26, 1.09, 0.76, 0.95, 1.53, 1.50, 1.32, 1.09, 0.83, 0.69, 1.02, 0.96, 0.20],
        [0.55, 1.00, 1.14, 0.94, 0.73, 0.88, 0.82, 1.23, 1.35, 1.27, 0.89, 0.67, 0.94, 0.81, 0.80],
        [0.29, 0.54, 0.57, 0.77, 0.97, 0.93, 0.57, 0.80, 1.32, 1.87, 0.61, 0.80, 0.61, 0.59, 0.57],
        [0.33, 0.38, 0.40, 0.41, 0.44, 0.85, 0.60, 0.61, 0.71, 0.95, 0.74, 1.06, 0.59, 0.56, 0.57],
        [0.31, 0.21, 0.25, 0.33, 0.39, 0.53, 0.68, 0.53, 0.55, 0.51, 0.82, 1.17, 0.85, 0.85, 0.33],
        [0.26, 0.25, 0.19, 0.24, 0.19, 0.34, 0.40, 0.39, 0.47, 0.55, 0.41, 0.78, 0.65, 0.85, 0.57],
        [0.09, 0.11, 0.12, 0.20, 0.19, 0.22, 0.13, 0.30, 0.23, 0.13, 0.21, 0.28, 0.36, 0.70, 0.60],
        [0.14, 0.15, 0.21, 0.10, 0.24, 0.17, 0.15, 0.41, 0.50, 0.71, 0.53, 0.76, 0.47, 0.74, 1.47],
    ]
)
BANDS = (*range(0, 75, 5), math.inf)  # 0-4, 5-9, ..., 65-69, 70+
UK_2006 = np.array(
    [3458060, 3556024, 3824317, 3960916, 3911291, 3762213, 4174675, 4695853,
     4653082, 3986098, 3620216, 3892985, 3124676, 2706365, 6961183],
    dtype=float,
)

polymod = ContactMatrix.from_array(PUBLISHED.T, BANDS, source_population=UK_2006)
polymod

## A. The survey loads the right way round

The published table is contact-by-participant, so it is transposed once on the
way in (`PUBLISHED.T`). The heatmap is `polymod.to_pandas()`: participant band
down the side, contact band along the top. The line chart compares the two
sums that are easy to confuse: `total()` (row sums — contacts each participant
reported) and the column sums (how often each band *was reported as a
contact*). Chapter 17 of the textbook points out that the second depends on
how many people are in the contacted band.

In [ ]:
heat = polymod.to_pandas().plot(
    kind="imshow",
    title="A. POLYMOD Great Britain: contacts per participant per day",
    labels={"x": "contact band", "y": "participant band", "color": "contacts"},
)
heat.show()

sums = pd.DataFrame(
    {
        "total(): contacts reported per participant (row sum)": polymod.total(),
        "column sum: times reported as a contact": polymod.values.sum(axis=0),
    },
    index=pd.Index(polymod.labels, name="band"),
)
fig = sums.plot(title="A. Row sums and column sums are different quantities", markers=True)
fig.update_layout(xaxis_title="age band", yaxis_title="contacts per day", legend_title="")
fig.show()
sums.round(2).T

> **What to check**
>
> - The heatmap has **participant** bands down the side. Its brightest cells are on the diagonal at `[5,10)`, `[10,15)` and `[15,20)` (about 6.6–6.9 contacts a day with their own age), and a fainter stripe runs parallel to the diagonal about 30 years off it (parents and children).
> - In the line chart the row sums peak at `[15,20)` (about **15.5**) and are lowest for `[70,inf)` (about **7.3**).
> - The column sums instead **jump up** at the last band, from about **3.9** at `[65,70)` to about **6.8** at `[70,inf)`: the open-ended 70+ band has many more people in it to be contacted. The row sums show no such jump.

## B. Frames and per-setting stacks load to the same matrix

Survey exports are usually long frames, one row per (participant band, contact
band). Below the same numbers are written as one, with `"0-4"`-style labels,
and read back with `ContactMatrix.from_frame`; the rows are shuffled first, so
the loader has to order the bands itself.

Surveys also split contacts by setting. To have a per-setting frame to load,
the next cell splits the matrix into an **illustrative** `"school"` layer —
*an assumption for this page, not survey data*: 60% of contacts among the three
school-age bands `[5,20)` — and `"other"`, the rest. `SettingStack.from_frame`
reads the frame with its `setting` column; `total()` sums the settings back.
The bars show each setting's contacts per participant, stacked; the line is
the all-settings `total()` from part A.

In [ ]:
labels = [f"{lo}-{lo + 4}" for lo in range(0, 70, 5)] + ["70+"]
k = len(labels)
long = pd.DataFrame(
    {
        "participant_age": [labels[i] for i in range(k) for _ in range(k)],
        "contact_age": [labels[j] for _ in range(k) for j in range(k)],
        "contacts": polymod.values.reshape(-1),
    }
).sample(frac=1.0, random_state=1)
from_frame = ContactMatrix.from_frame(
    long, row="participant_age", col="contact_age", value="contacts", source_population=UK_2006
)

school_values = np.zeros_like(polymod.values)
school_values[1:4, 1:4] = 0.6 * polymod.values[1:4, 1:4]
per_setting = pd.concat(
    [
        long.assign(setting="school", contacts=school_values.reshape(-1)[long.index]),
        long.assign(setting="other", contacts=(polymod.values - school_values).reshape(-1)[long.index]),
    ]
)
stack = SettingStack.from_frame(
    per_setting, row="participant_age", col="contact_age", source_population=UK_2006
)

by_setting = pd.DataFrame(
    {name: stack[name].total() for name in stack}, index=pd.Index(polymod.labels, name="band")
)
fig = by_setting.plot(kind="bar", title="B. Contacts per participant, stacked by setting")
fig.add_scatter(x=list(polymod.labels), y=polymod.total(), mode="lines+markers", name="all settings (part A)")
fig.update_layout(xaxis_title="participant band", yaxis_title="contacts per day", barmode="stack")
fig.show()

pd.Series(
    {
        "from_frame == from_array": from_frame == polymod,
        "max |from_frame - from_array|": float(np.max(np.abs(from_frame.values - polymod.values))),
        "settings": list(stack),
        "max |stack.total() - from_array|": float(np.max(np.abs(stack.total().values - polymod.values))),
    },
    name="value",
).to_frame()

> **What to check**
>
> - `from_frame == from_array` is **True** and both differences in the table are exactly **0.0**: the shuffled long frame, with `"0-4"` / `"70+"` labels, loads to the same matrix.
> - The settings are `['school', 'other']`, and in the chart the stacked bars reach the all-settings line exactly in every band. The school layer appears only in the `[5,10)`, `[10,15)` and `[15,20)` bars.

## C. Bad input is refused, naming the offending value

Each row below is a broken input. The table shows the `ValueError` message
each one raises.

In [ ]:
bad_values = polymod.values.copy()
bad_values[3, 7] = -0.2
attempts = {
    "not square": lambda: ContactMatrix.from_array(polymod.values[:, :-1], BANDS),
    "one band edge too many": lambda: ContactMatrix.from_array(polymod.values, (*BANDS[:-1], 75, math.inf)),
    "bands out of order": lambda: ContactMatrix.from_array(polymod.values, (0, 5, 15, 10, *BANDS[4:])),
    "a negative contact rate": lambda: ContactMatrix.from_array(bad_values, BANDS),
    "population of the wrong length": lambda: ContactMatrix.from_array(
        polymod.values, BANDS, source_population=UK_2006[:-1]
    ),
    "an unreadable band label": lambda: ContactMatrix.from_frame(
        long.replace({"participant_age": {"15-19": "teens"}, "contact_age": {"15-19": "teens"}}),
        row="participant_age",
        col="contact_age",
    ),
    "a missing (participant, contact) pair": lambda: ContactMatrix.from_frame(
        long.iloc[1:], row="participant_age", col="contact_age"
    ),
}


def message(attempt):
    try:
        attempt()
    except ValueError as error:
        return str(error)
    return "accepted (this would be a failure)"


pd.set_option("display.max_colwidth", None)
pd.DataFrame({"message": {name: message(attempt) for name, attempt in attempts.items()}})

> **What to check** — every row is refused, and its message names the actual value:
>
> - *not square* names the shape `(15, 14)`;
> - *one band edge too many* says **17** band edges for a 15 × 15 matrix, expected **16**;
> - *bands out of order* names the pair `bands[2] = 15 then bands[3] = 10`;
> - *a negative contact rate* names `values[3, 7] = -0.2`;
> - *population of the wrong length* names shape `(14,)` against the expected `(15,)`;
> - *an unreadable band label* names `'teens'`;
> - *a missing pair* names the pair that is missing.

## D. Reciprocity and mean contacts are measurable

A contact is between two people, so in a closed population the total contacts
band `i` reports with band `j` should equal the total `j` reports with `i`:
$c_{ij} N_i = c_{ji} N_j$. A survey only ever estimates that. The heatmap is
`polymod.reciprocity_error().matrix`, $c_{ij} N_i - c_{ji} N_j$ in millions of
contacts a day, against the UK 2006 population the survey carries as
`source_population`. `reciprocity_error().relative` is one number for the
whole matrix, and `mean_contacts()` is the population-weighted average
contacts per person — the headline number a survey paper reports.

In [ ]:
reciprocity = polymod.reciprocity_error()
error = pd.DataFrame(
    reciprocity.matrix / 1e6,
    index=pd.Index(polymod.labels, name="participant"),
    columns=pd.Index(polymod.labels, name="contact"),
)
fig = error.plot(
    kind="imshow",
    title="D. Reciprocity error c_ij N_i - c_ji N_j (millions of contacts a day)",
    labels={"x": "contact band", "y": "participant band", "color": "millions"},
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0.0,
)
fig.show()

pd.Series(
    {
        "relative reciprocity error": round(reciprocity.relative, 3),
        "largest |error| (millions/day)": round(float(np.max(np.abs(error.values))), 2),
        "at (participant, contact)": error.abs().stack().idxmax(),
        "mean contacts per person per day": round(polymod.mean_contacts(), 2),
    },
    name="value",
).to_frame()

> **What to check**
>
> - The heatmap is **antisymmetric**: each cell is the negative of its mirror across the diagonal, and the diagonal is zero (white).
> - The last row (`[70,inf)` participants) is mostly red and the last column mostly blue: people aged 70+ report more contacts with younger bands than those bands report with them. The largest error, about **3.99** million a day, is between `[70,inf)` and `[15,20)`.
> - The relative reciprocity error is about **0.25** — this raw survey matrix is far from reciprocal.
> - Mean contacts per person per day is about **11.07**.